# Email Spam Detection with Machine Learning

Run the cells in order. Put `spam.csv` in the notebook's working directory. The first run may need internet access to download NLTK stopwords.


## Imports and setup


In [ ]:
from pathlib import Path
import pickle
import re

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

try:
    english_stopwords = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    english_stopwords = set(stopwords.words('english'))

ps = PorterStemmer()


## Load and inspect the dataset

The CSV should contain `v1` labels (`ham` / `spam`) and `v2` message text.


In [ ]:
data_path = Path('spam.csv')
if not data_path.is_file():
    raise FileNotFoundError('Place spam.csv in the notebook working directory.')

spam = pd.read_csv(data_path, encoding='ISO-8859-1')
missing = {'v1', 'v2'} - set(spam.columns)
if missing:
    raise ValueError(f'Missing expected CSV columns: {sorted(missing)}')
spam = spam[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'message'})
spam['label'] = spam['label'].astype('string').str.strip().str.lower()
spam['message'] = spam['message'].fillna('').astype(str)
if spam['label'].isna().any() or not spam['label'].isin(['ham', 'spam']).all():
    raise ValueError("Expected labels to be 'ham' or 'spam'.")

display(spam.head())
print('Dataset shape:', spam.shape)
print('Null values:', spam.isnull().sum())
print('Class counts:\n', spam['label'].value_counts())


In [ ]:
spam['label'].value_counts().plot(kind='bar', title='Message counts by label')
plt.xlabel('Label')
plt.ylabel('Number of messages')
plt.show()


## Preprocessing and feature extraction

Split before fitting the vectorizer so the vocabulary is learned from training data only.


In [ ]:
def preprocess_message(message):
    words = re.sub(r'[^a-zA-Z]', ' ', str(message)).lower().split()
    return ' '.join(ps.stem(word) for word in words if word not in english_stopwords)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    spam['message'], spam['label'], test_size=0.20, random_state=42, stratify=spam['label']
)
X_train_clean = X_train_raw.map(preprocess_message)
X_test_clean = X_test_raw.map(preprocess_message)

cv = CountVectorizer(max_features=4000)
X_train = cv.fit_transform(X_train_clean)
X_test = cv.transform(X_test_clean)
y_train_binary = y_train.map({'ham': 0, 'spam': 1})
y_test_binary = y_test.map({'ham': 0, 'spam': 1})
print('Training matrix:', X_train.shape, 'Test matrix:', X_test.shape)


## Train and evaluate classifiers


In [ ]:
models = {
    'Random Forest Classifier': RandomForestClassifier(random_state=42),
    'Decision Tree Classifier': DecisionTreeClassifier(random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(),
}
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train_binary)
    predictions[name] = model.predict(X_test)
    print(f'\n{name}')
    print('Confusion matrix (rows=true, columns=predicted; 0=ham, 1=spam):')
    print(confusion_matrix(y_test_binary, predictions[name], labels=[0, 1]))
    print(f'Accuracy: {accuracy_score(y_test_binary, predictions[name]):.4f}')
    print(classification_report(y_test_binary, predictions[name], labels=[0, 1], target_names=['ham', 'spam'], zero_division=0))


In [ ]:
best_model = models['Multinomial Naive Bayes']
cm = confusion_matrix(y_test_binary, predictions['Multinomial Naive Bayes'], labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Multinomial Naive Bayes confusion matrix')
plt.show()


## Predict new messages and save model

The saved pickle contains both the fitted classifier and its vectorizer. Only load pickle files you trust.


In [ ]:
sample_messages = [
    'URGENT: Your account has been suspended! Click here immediately to verify your billing information before permanent deletion.',
    'Hi Sarah, are we still meeting for lunch tomorrow at 1 PM? Let me know if that time still works for you or if we need to reschedule.'
]
sample_clean = [preprocess_message(message) for message in sample_messages]
sample_vectors = cv.transform(sample_clean)
sample_predictions = best_model.predict(sample_vectors)
for message, prediction in zip(sample_messages, sample_predictions):
    print('Example message:', message)
    print('Model prediction:', 'Spam' if prediction == 1 else 'Not spam', '\n')

with open('spam_model.pkl', 'wb') as model_file:
    pickle.dump({'model': best_model, 'vectorizer': cv}, model_file)
print('Saved spam_model.pkl')
